# J2 | API & Prompt Engineering

## Ou les prompt heureux...

**Instructeur :** Mickael Temporão

**Durée estimée :** 2-3 heures

**Objectifs d'apprentissage :**
1. Comprendre ce qu'est une API et comment interroger un serveur avec Python.
2. Découvrir et prendre en main l'API d'un Modèle de Langage (LLM).
3. Maîtriser les principes du *Prompt Engineering* appliqués aux sciences sociales (Zero-shot, Few-shot).
4. Appliquer ces concepts pour annoter et classifier un petit jeu de données de citations politiques.

**Prérequis :**
- Avoir un compte Google (pour Colab).
- Avoir créé une clé API OpenAI (ou utiliser la clé fournie pendant l'atelier).
- Notions basiques de Python (J1)

## 0. Installation et configuration de l'environnement

Avant de commencer, nous devons installer les bibliothèques Python nécessaires pour interagir avec le web (`requests`), manipuler des données (`pandas`) et interroger les LLMs (`openai`).

In [ ]:
!pip install requests pandas openai

## 1. Qu'est-ce qu'une API ? (Concept et Pratique)

En sciences sociales, on entend souvent parler d'API pour extraire des données de Twitter/X, Wikipedia ou des journaux. 

**API** signifie *Application Programming Interface* (Interface de Programmation d'Application). 
L'analogie la plus connue est celle du **restaurant** :
- Vous (l'utilisateur ou votre code Python) êtes le client à la table.
- La cuisine (le serveur web ou la base de données) prépare les plats (les données).
- **L'API, c'est le serveur (waiter)** : vous lui donnez votre commande (la *requête* ou *request*), il la transmet à la cuisine, et il vous rapporte votre plat (la *réponse*, généralement en format `JSON`).

Faisons une requête HTTP basique pour demander à une API publique (l'API de recherche d'OpenData) de nous renvoyer une information.

In [ ]:
import requests
import json

# L'adresse à laquelle on passe notre commande (l'URL de l'API)
url = "https://data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/catalog/datasets"

# On définit nos paramètres (notre commande spécifique)
params = {
    "limit": 1  # On demande un seul résultat
}

# Le serveur HTTP apporte notre commande à la cuisine
response = requests.get(url, params=params)

# On vérifie si la commande a bien été reçue (Code 200 = OK)
print(f"Statut de la réponse : {response.status_code}")

# On affiche le 'plat' qu'il nous a ramené, formaté en JSON
data = response.json()
print("Données reçues :")
print(json.dumps(data, indent=2, ensure_ascii=False)[:500] + "\n... [contenu tronqué]")

### Hack-Time 🛠️

Essayez de modifier la requête ci-dessus. L'API d'OpenData du gouvernement français accepte un paramètre `where` pour faire une recherche textuelle.
Ajoutez à `params` une ligne pour chercher le mot "sociologie" ou "politique". Par exemple : `"where": "search(title, 'sociologie')"`.

*Pour rester simple, essayez de demander `limit: 5` au lieu de 1 et observez la taille de la réponse JSON.*

In [ ]:
# À vous de jouer : copiez-collez le code ci-dessus et modifiez 'limit' ou ajoutez de nouveaux paramètres !



## 2. Introduction aux API de LLM (OpenAI)

Maintenant que nous savons envoyer une commande à un serveur pour recevoir du texte, appliquons cela aux Modèles de Langage.
Lorsqu'on utilise ChatGPT via le site web, l'interface graphique envoie en arrière-plan nos requêtes à l'API d'OpenAI. Nous allons faire cette étape directement en Python.

**Sécurité importante :** Ne mettez **jamais** votre clé API directement dans le code ! 
Sur Google Colab, utilisez le panneau "Secrets" (l'icône de clé à gauche) pour stocker une variable `OPENAI_API_KEY`.

In [ ]:
import os
from openai import OpenAI

# Sur Google Colab, on récupère la clé secrète via userdata
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    # Fallback pour un environnement local
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("ATTENTION : Aucune clé API trouvée. Veuillez configurer OPENAI_API_KEY dans les secrets Colab.")
else:
    # On instancie notre "serveur" (le client API)
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")

Pour parler à l'API OpenAI, il faut structurer les messages. Contrairement à ChatGPT où on tape un texte brut, via l'API, on définit explicitement **les rôles** :
- `system` : Les instructions générales (le cadrage, le "rôle" du modèle).
- `user` : Le texte ou la question de l'utilisateur (la donnée à analyser).

Créons une petite fonction de base pour interroger le modèle.

In [ ]:
def analyze_text(text, system_prompt, model="gpt-4o-mini", temperature=0.2):
    """
    Fonction simple pour envoyer un texte et un prompt système à l'API OpenAI.
    """
    if not api_key:
        return "Erreur: Clé API manquante."
        
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            temperature=temperature,  # 0 = très prévisible/déterministe, 1 = créatif/aléatoire
            max_tokens=250 # On limite la longueur pour réduire les coûts et accélérer
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erreur lors de l'appel à l'API : {e}"

# Testons la fonction !
consigne = "Tu es un assistant sarcastique. Réponds en une seule phrase."
mon_texte = "Peux-tu m'expliquer ce qu'est la démocratie ?"

print("Réponse du LLM :")
print(analyze_text(mon_texte, consigne, temperature=0.8))

## 3. Le Prompt Engineering comme Méthodologie de Recherche?

Pour un politiste ou un sociologue, **le prompt est l'équivalent d'un protocole d'annotation (codebook)**. Si le prompt est ambigu, le codage de vos données sera mauvais.

**Attention** : Dans tous les cas, il faudra vérifier dérrière!

Explorons trois techniques fondamentales pour la recherche en SHS.

### A. Zero-Shot Classification
On demande au modèle de classifier un texte sans lui donner aucun exemple de texte déjà annoté. C'est l'approche la plus basique.

In [ ]:
text_to_analyze = "Le gouvernement doit intervenir massivement pour protéger nos services publics et encadrer la finance."

zero_shot_prompt = """
Tu es un expert en idéologie politique française.
Lis le texte suivant et détermine son orientation idéologique parmi ces 3 catégories :
- GAUCHE
- DROITE
- CENTRE

Réponds UNIQUEMENT par le mot de la catégorie, sans aucune justification ni ponctuation.
"""

print("Classification Zero-Shot :")
print(analyze_text(text_to_analyze, zero_shot_prompt, temperature=0.0))

### B. Few-Shot Classification
Les LLMs fonctionnent beaucoup mieux "par l'exemple". Le *Few-Shot* consiste à inclure 2 ou 3 exemples directement dans le prompt système pour montrer au modèle exactement le format et le raisonnement attendu. Cela augmente drastiquement la fiabilité et la validité de la mesure.

In [ ]:
few_shot_prompt = """
Tu es un expert en idéologie politique française.
Classe les textes selon 3 catégories : GAUCHE, DROITE, CENTRE.
Réponds UNIQUEMENT par la catégorie.

Exemples :
Texte : "Il faut baisser les charges des entreprises pour libérer l'économie et valoriser le travail."
Catégorie : DROITE

Texte : "L'urgence est d'augmenter le SMIC et de taxer les super-profits pour plus de justice sociale."
Catégorie : GAUCHE

Texte : "Le pragmatisme exige de dépasser les vieux clivages et d'allier libéralisme économique et progressisme sociétal."
Catégorie : CENTRE

À toi de jouer sur ce nouveau texte :
"""

print("Classification Few-Shot :")
print(analyze_text(text_to_analyze, few_shot_prompt, temperature=0.0))

### C. Chain of Thought (CoT)
Parfois, la classification est complexe. La technique *Chain of Thought* (Chaîne de Pensée) consiste à demander au modèle de "penser à voix haute" et de justifier son choix **avant** de donner l'étiquette finale. Cela réduit les hallucinations et améliore la logique.

In [ ]:
cot_prompt = """
Tu es un politiste expert.
Analyse le texte pour déterminer s'il est de GAUCHE, DROITE ou CENTRE.

Procède en deux étapes :
1. Raisonnement : Analyse brièvement les mots-clés et l'argumentation.
2. Label final : Termine ta réponse par "LABEL: [Catégorie]".
"""

print("Classification avec Chain of Thought :")
print(analyze_text(text_to_analyze, cot_prompt, temperature=0.2))

### Hack-Time 🛠️

Modifiez le texte `text_to_analyze` pour utiliser une phrase volontairement ambiguë ou populiste (ex: *"Le système actuel écrase le peuple, il faut rendre le pouvoir aux citoyens face aux élites mondialisées."*).

Testez cette phrase avec le `zero_shot_prompt`, puis avec le `cot_prompt`. Observez-vous une différence ? Modifiez la `temperature` à 1.0 et testez plusieurs fois : que remarquez-vous sur la stabilité des réponses ?

**Défi optionnel**
Modifiez le prompt Few-Shot pour rajouter un exemple ambigu.

In [ ]:
# Écrivez votre code ici



## 4. Atelier Pratique : Labellisation de données politiques

En SHS, on analyse rarement un seul texte, mais plutôt un corpus entier (des centaines ou des milliers d'observations).
Construisons un petit corpus d'exemple à l'aide de la librairie `pandas`.

In [ ]:
import pandas as pd

# Notre mini-dataset fictif (citations politiques)
data = [
    {"id": 1, "text": "La priorité absolue est la sécurité dans nos quartiers et une baisse massive de l'immigration."},
    {"id": 2, "text": "Je propose une grande marche pour le climat et un revenu étudiant universel."},
    {"id": 3, "text": "Nous devons flexibiliser le marché du travail tout en protégeant les plus vulnérables. C'est le 'en même temps'."},
    {"id": 4, "text": "Selon les derniers sondages, 60% des Français sont inquiets pour leur pouvoir d'achat."},
    {"id": 5, "text": "Aujourd'hui à l'Assemblée, le débat sur le budget a été très houleux, nous rapporte notre envoyé spécial."}
]

df = pd.DataFrame(data)
display(df)

**Mission pour cet atelier :**
Vous voulez ajouter deux variables (colonnes) à votre jeu de données :
1. **L'idéologie** (Gauche, Droite, Centre, Neutre/NA)
2. **Le type de locuteur supposé** (Personnalité politique, Journaliste)

Pour automatiser cela, nous allons écrire une fonction qui va appliquer notre LLM sur chaque ligne de notre tableau.

In [ ]:
# On prépare notre prompt de codage (Codebook)
system_prompt_codage = """
Tu es un assistant de recherche en science politique. Ton travail est d'annoter des textes courts.

Pour chaque texte, tu dois extraire 2 variables au format CSV précis :
Ideologie,Locuteur

Ideologie doit être : GAUCHE, DROITE, CENTRE, ou NEUTRE.
Locuteur doit être : POLITIQUE ou JOURNALISTE.

Réponds STRICTEMENT selon ce format : Ideologie,Locuteur
N'ajoute aucun autre texte, pas d'explication.

Exemple 1 :
Texte : "La bourse a clôturé en baisse de 2 points ce soir."
Réponse : NEUTRE,JOURNALISTE

Exemple 2 :
Texte : "Taxons les riches pour financer l'hôpital !"
Réponse : GAUCHE,POLITIQUE
"""

### Hack-Time 🛠️

Complétez le code ci-dessous. Nous voulons appliquer la fonction `analyze_text` à la colonne `text` de notre DataFrame.

*Astuce : Vous pouvez utiliser une boucle `for` sur les lignes du DataFrame, ou la méthode `.apply()` de Pandas.*

In [ ]:
# Complétez le code !
resultats = []

# Pour chaque texte dans df["text"]...
for texte in df["text"]:
    # 1. Appeler la fonction analyze_text avec system_prompt_codage
    # 2. Stocker la réponse dans la liste 'resultats'
    pass # <-- Remplacez "pass" par votre code

# --- Ne touchez pas au code ci-dessous (sauf si vous savez ce que vous faites !) ---
'''
# Quand votre boucle fonctionnera, décommentez ceci pour ranger les résultats dans le DataFrame :
df["LLM_Reponse"] = resultats

# On peut même séparer la réponse CSV en deux colonnes propres
df[["Ideologie_LLM", "Locuteur_LLM"]] = df["LLM_Reponse"].str.split(",", expand=True)
display(df)
'''

## Conclusion : Limites Méthodologiques et Éthiques

Bravo ! Vous venez d'utiliser un LLM pour coder un jeu de données. 
Cependant, dans un contexte de recherche publié, il faut garder un regard critique :

1. **Reproductibilité :** L'API évolue. Le modèle `gpt-4o-mini` d'aujourd'hui ne donnera peut-être pas les mêmes réponses dans un an. En SHS, il est recommandé de préciser la date de collecte et la version exacte du modèle.
2. **Fiabilité et Validité :** Les modèles ont des biais. Ils tendent souvent vers des définitions américaines de la gauche et de la droite (biais d'alignement RLHF).
3. **Double Codage :** Idéalement, on ne fait pas aveuglément confiance au LLM. On procède à un "accord inter-juges" : un chercheur humain code manuellement 10% de l'échantillon, et on calcule un taux de fiabilité (ex: *Krippendorff's alpha*) entre l'humain et l'IA.

*Fin de la session PM. N'hésitez pas à poser vos questions ou à expérimenter avec vos propres données !*